# Tuning Systems

In [ ]:
import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy
import scipy
import scipy.signal

from mirdotcom import mirdotcom

mirdotcom.init()

## Introduction

### Tuning Systems

In twelve-tone **equal temperament** ([Wikipedia](https://en.wikipedia.org/wiki/Equal_temperament)), all twelve semitones within the octave have the same width. With this tuning system, expressed as a frequency ratio, the interval of one semitone is $2^{1/12}$. Expressed in **cents**, this same interval is defined to be 100 cents. Therefore, the octave has 1200 cents.

In **just intonation** ([Wikipedia](https://en.wikipedia.org/wiki/Just_intonation)), the frequency ratio is expressed as a fraction between two small integers, e.g. 3:2, 4:3. As a result, the higher harmonic partials between two notes will overlap, resulting in a consonant interval that is pleasing to the ear. In **5-limit just tuning**, these fractions are expressed with prime factors no larger than 5, i.e. {2, 3, 5}. In **7-limit just tuning**, these fractions are expressed with prime factors no larger than 7, i.e. {2, 3, 5, 7}. For example, 7:4 is a 7-limit interval, but it is not a 5-limit interval.

In **Pythagorean tuning** ([Wikipedia](https://en.wikipedia.org/wiki/Pythagorean_tuning)), every frequency ratio is based upon the ratio 3:2. To find that ratio, from one note in the interval, step around the Circle of Fifths until you reach the other note in the interval, multiplying (if stepping forward) or dividing (if stepping backward) by 3/2 with each step. Finally, multiply or divide by 2 enough times to return to the octave of interest. Pythagorean tuning can also be considered **3-limit just tuning** since every ratio only uses prime factors no greater than 3.

### How To Use This Notebook

In the examples below, listen to each interval, and compare intervals both visually and aurally across tuning systems. In some tuning systems, the upper harmonics do not align. In such cases, try to listen for dissonance and beat frequencies. Then compare them to the tuning systems where the harmonics align perfectly.

### Notebook Setup

Global parameters for this notebook:

In [ ]:
T = 5  # duration in seconds
sr = 22050  # sampling rate in Hertz
fcutoff = 4000  # frequency cutoff for filter

Functions used to create the sounds and figures:

In [ ]:
def simulate_tone(f0, harmonics=None):
    """Returns a tone with a specified fundamental frequency and harmonic amplitudes."""
    if harmonics is None:
        harmonics = [1]
    t = numpy.linspace(0, T, T * sr, endpoint=False)
    x = sum(
        v * numpy.sin(2 * numpy.pi * i * f0 * t) for i, v in enumerate(harmonics, 1)
    )
    return x

In [ ]:
def filter_tone(x, fcutoff=None):
    """Return a low-pass filtered signal."""
    if fcutoff is None:
        fcutoff = sr / 2.0
    h = scipy.signal.firwin(55, 2 * float(fcutoff) / sr)
    y = scipy.signal.convolve(x, h)
    return y

In [ ]:
def make_double_stop(ratio, f0=440, harmonics=None):
    """Listen to two tones played simultaneously, and plot the tones' spectra."""
    if harmonics is None:
        harmonics = [1]
    f1 = ratio * f0

    # Generate both tones.
    x0 = filter_tone(simulate_tone(f0, harmonics=harmonics), fcutoff=fcutoff)
    x1 = filter_tone(simulate_tone(f1, harmonics=harmonics), fcutoff=fcutoff)

    # Add both tones, and normalize.
    y = x0 + x1
    y = 0.5 * y / y.max()

    # Generate both spectra.
    X0 = scipy.fft.fft(x0)
    X1 = scipy.fft.fft(x1)

    # Create frequency variable.
    N = len(X0)
    f = numpy.linspace(0, sr, N, endpoint=False)

    # Plot spectrum of both notes.
    plt.semilogy(f, abs(X0), marker="o", linewidth=1)
    plt.semilogy(f, abs(X1), color="r", linewidth=1)

    plt.xlim(xmin=0, xmax=(len(harmonics) + 1) * (f1 if ratio > 1 else f0))
    plt.ylim(ymin=0.1)

    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")
    plt.legend(("f0 = %.2f Hz" % f0, "f0 = %.2f Hz" % f1))

    # Output audio widget.
    print("ratio between notes:", ratio)
    print("difference in cents:", numpy.log2(ratio) * 1200)
    return ipd.Audio(y, rate=sr)

## Unison

*Within each section below, the intervals are provided in order of interval width from lowest to highest.*

In [ ]:
make_double_stop(1)

## Octaves

In [ ]:
harmonics = [1.0, 0.5]

Just intonation or equal temperament, 12 semitones, 1200 cents:

In [ ]:
make_double_stop(2, harmonics=harmonics)

Pythagorean tuning, twelve steps forward on the Circle of Fifths. In Pythagorean tuning, multiply the fundamental frequency by 3/2 twelve times, and then divide by two enough times to return to the octave of interest:

$$ \left( \frac{3}{2} \right)^{12} \left( \frac{1}{2} \right)^6 = \frac{531441}{262144} \approx 2.0273 $$

In [ ]:
make_double_stop(531441.0 / 262144, harmonics=harmonics)

The *Pythagorean comma*, the degree of inconsistency when trying to define a twelve-tone scale using only perfect fifths, is about 1.0136 when expressed as a frequency ratio:

$$ \left( \frac{3}{2} \right)^{12} \left( \frac{1}{2} \right)^7 = \frac{531441}{524288} \approx 1.0136 $$

## Fifths

In [ ]:
harmonics = [1.0, 0.1, 0.1]

Equal temperament, seven semitones, 700 cents:

In [ ]:
make_double_stop(2 ** (7.0 / 12), harmonics=harmonics)

Just intonation or Pythagorean tuning, one step forward on the Circle of Fifths:

In [ ]:
make_double_stop(3.0 / 2, harmonics=harmonics)

## Fourths

In [ ]:
harmonics = [1.0, 0.01, 0.1, 0.1]

Just intonation or Pythagorean tuning, one step backward on the Circle of Fifths:

$$ \left( \frac{2}{3} \right) \left( \frac{2}{1} \right) = \frac{4}{3} = 1.\overline{3} $$

In [ ]:
make_double_stop(4.0 / 3, harmonics=harmonics)

Equal temperament, five semitones:

In [ ]:
make_double_stop(2 ** (5.0 / 12), harmonics=harmonics)

## Major Sixths

In [ ]:
harmonics = [1.0, 0.01, 0.1, 0.01, 0.1]

Just intonation:

In [ ]:
make_double_stop(5.0 / 3, harmonics=harmonics)

Equal temperament, i.e. nine semitones:

In [ ]:
make_double_stop(2 ** (9.0 / 12), harmonics=harmonics)

Pythagorean tuning, three steps forward on the Circle of Fifths:

$$ \left( \frac{3}{2} \right)^3 \left( \frac{1}{2} \right) = \frac{27}{16} = 1.6875 $$

In [ ]:
make_double_stop(27.0 / 16, harmonics=harmonics)

## Minor Thirds

In [ ]:
harmonics = [1.0, 0.01, 0.01, 0.01, 0.1, 0.1]

Pythagorean tuning, three steps backward on the Circle of Fifths:

$$ \left( \frac{2}{3} \right)^3 \left( \frac{2}{1} \right)^2 = \frac{32}{27} = 1.\overline{185} $$

In [ ]:
make_double_stop(32.0 / 27, harmonics=harmonics)

Equal temperament, three semitones:

In [ ]:
make_double_stop(2 ** (3.0 / 12), harmonics=harmonics)

Just intonation:

In [ ]:
make_double_stop(6.0 / 5, harmonics=harmonics)

## Major Thirds

In [ ]:
harmonics = [1.0, 0.01, 0.01, 0.1, 0.1]

Just intonation:

In [ ]:
make_double_stop(5.0 / 4, harmonics=harmonics)

Equal temperament, four semitones:

In [ ]:
make_double_stop(2 ** (4.0 / 12), harmonics=harmonics)

Pythagorean tuning, four steps forward on the Circle of Fifths:

$$ \left( \frac{3}{2} \right)^4 \left( \frac{1}{2} \right)^2 = \frac{81}{64} = 1.265625 $$

In [ ]:
make_double_stop(81.0 / 64, harmonics=harmonics)

## Minor Sixths

In [ ]:
harmonics = [1.0, 0.001, 0.001, 0.001, 0.01, 0.001, 0.001, 0.01]

Pythagorean tuning, four steps backward on the Circle of Fifths:

$$ \left( \frac{2}{3} \right)^4 \left( \frac{2}{1} \right)^3 = \frac{128}{81} \approx 1.58 $$

In [ ]:
make_double_stop(128.0 / 81, harmonics=harmonics)

Equal temperament, eight semitones:

In [ ]:
make_double_stop(2 ** (8.0 / 12), harmonics=harmonics)

Just intonation:

In [ ]:
make_double_stop(8.0 / 5, harmonics=harmonics)

## Major Seconds

In [ ]:
harmonics = [1.0, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.1, 0.1, 0.1]

In just intonation, there are two versions of the major second: the *major tone* a.k.a. *greater tone* (9:8), and the *minor tone* a.k.a. *lesser tone* (10:9). The difference between the two, expressed as a ratio of their widths, is called a *syntonic comma* and is equal to 81:80.

$$ \frac{10}{9} n = \frac{9}{8} \Rightarrow n = \frac{81}{80} $$

Just intonation (minor tone a.k.a. lesser tone). Observe how the tenth harmonic of the lower note aligns with the ninth harmonic of the upper note.

In [ ]:
make_double_stop(10.0 / 9, harmonics=harmonics)

Equal temperament, two semitones:

In [ ]:
make_double_stop(2 ** (2.0 / 12), harmonics=harmonics)

Just intonation (major tone a.k.a. greater tone) or Pythagorean tuning, two steps forward on the Circle of Fifths:

$$ \left( \frac{3}{2} \right)^2 \left( \frac{1}{2} \right) = \frac{9}{8} = 1.125 $$

Observe how the ninth harmonic of the lower note aligns with the eighth harmonic of the upper note.

In [ ]:
make_double_stop(9.0 / 8, harmonics=harmonics)

## Minor Sevenths

In [ ]:
harmonics = [1.0, 0.01, 0.01, 0.1, 0.1, 0.01, 0.1, 0.01, 0.1]

7-limit just intonation, a.k.a. septimal minor seventh, harmonic seventh, or subminor seventh:

In [ ]:
make_double_stop(7.0 / 4, harmonics=harmonics)

Pythagorean tuning, two steps backward on the Circle of Fifths, or small just minor seventh:

$$ \left( \frac{2}{3} \right)^2 \left( \frac{2}{1} \right)^2 = \frac{16}{9} = 1.\overline{7} $$

In [ ]:
make_double_stop(16.0 / 9, harmonics=harmonics)

Equal temperament, ten semitones:

In [ ]:
make_double_stop(2 ** (10.0 / 12), harmonics=harmonics)

5-limit just intonation, a.k.a. large just minor seventh:

In [ ]:
make_double_stop(9.0 / 5, harmonics=harmonics)

## Major Sevenths

In [ ]:
harmonics = [
    1.0,
] * 15

Just intonation:

In [ ]:
make_double_stop(15.0 / 8, harmonics=harmonics)

Equal temperament, eleven semitones:

In [ ]:
make_double_stop(2 ** (11.0 / 12), harmonics=harmonics)

Pythagorean tuning, five steps forward on the Circle of Fifths:

$$ \left( \frac{3}{2} \right)^5 \left( \frac{1}{2} \right)^2 = \frac{243}{128} \approx 1.8984 $$

In [ ]:
make_double_stop(243.0 / 128, harmonics=harmonics)

## Minor Seconds

In [ ]:
harmonics = [
    0.01,
] * 16
harmonics[0] = 1.0
harmonics[14] = 0.1
harmonics[15] = 0.1

Pythagorean tuning, five steps backward on the Circle of Fifths:

$$ \left( \frac{2}{3} \right)^5 \left( \frac{2}{1} \right)^3 = \frac{256}{243} \approx 1.0535 $$

In [ ]:
make_double_stop(256.0 / 243, harmonics=harmonics)

Equal temperament, one semitone:

In [ ]:
make_double_stop(2 ** (1.0 / 12), harmonics=harmonics)

Just intonation:

In [ ]:
make_double_stop(16.0 / 15, harmonics=harmonics)

## Tritone, Augmented Fourths, Diminished Fifths

Diminished fifth, Pythagorean tuning, six steps backward on the Circle of Fifths:

$$ \left( \frac{2}{3} \right)^6 \left( \frac{2}{1} \right)^4 = \frac{1024}{729} \approx 1.40466 $$

In [ ]:
make_double_stop(1024.0 / 729)

Augmented fourth:

In [ ]:
make_double_stop(45.0 / 32)

Equal temperament, six semitones, 600 cents:

In [ ]:
make_double_stop(2 ** (6.0 / 12))

Diminished fifth:

In [ ]:
make_double_stop(64.0 / 45)

Augmented fourth, Pythagorean tuning, six steps forward on the Circle of Fifths:

$$ \left( \frac{3}{2} \right)^6 \left( \frac{1}{2} \right)^3 = \frac{729}{512} \approx 1.4238 $$

In [ ]:
make_double_stop(729.0 / 512)